# Invoice Region Detection and Business Parameter Extraction Using CNN, SSD, IoU, OCR, and Streamlit

**Member:** Hessam
**Role:** Project Manager, Solution Architect, Integration Lead, Streamlit App Lead

**Objective:** Validate all member outputs, integrate them into a final per-invoice JSON record, build/verify the Streamlit demo, and prepare the final report and presentation.

**Inputs expected:**
- All outputs from Rolando, Diana, Jordan, and Damir (see `../../model_interface_contract.md`)

**Outputs generated:**
- `outputs/final_json/sample_invoice_outputs/*.json`
- `outputs/reports/final_pipeline_report.md`
- `app/streamlit_app.py` (validated)
- `presentation/demo_script.md`

> Run this notebook top-to-bottom in Google Colab, or locally with the repo's virtualenv.
> Paths are resolved via `src/config.py` (pathlib-based) — never hardcode absolute local paths.


In [ ]:
# --- Google Colab setup cell ---
# If running in Colab: clone the repo (or mount Drive if you cloned there already) and
# install dependencies. Safe to skip locally if the repo is already on disk with deps installed.

import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/<your-org>/invoice-image-processing.git"  # TODO: set this
    REPO_DIR = "/content/invoice-image-processing"

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q -r requirements.txt")

    # Kaggle API credentials (upload kaggle.json when prompted) -- see dataset_sources.md
    from google.colab import files
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token):")
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        for fname in uploaded:
            os.replace(fname, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    print("Colab environment ready. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- assuming local repo checkout with requirements installed.")


In [ ]:
# --- Dataset path setup cell ---
# All paths go through src.config.PATHS (pathlib-based, no hardcoded absolute paths).

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PATHS, load_label_schema, load_required_fields

print("Repo root:", PATHS.repo_root)
print("Raw data dir:", PATHS.raw_dir)
print("Outputs dir:", PATHS.outputs_dir)

# If raw data isn't present yet, download it (see dataset_sources.md for kaggle.json setup):
#   python scripts/download_datasets.py --dataset all


In [ ]:
# --- Imports cell ---
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.image_preprocessing import preprocess_pipeline, to_grayscale, resize_image, denoise_image, threshold_image, deskew_image
from src.visualization import draw_boxes, show_image_grid
from src.annotation_utils import load_annotations, boxes_for_image
from src.iou import compute_iou, precision_recall_iou, evaluate_predictions_df


## 1. Validate that every member's output files exist

In [ ]:
import subprocess
result = subprocess.run(["python", str(PATHS.repo_root / "scripts" / "validate_dataset_paths.py")], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


## 2. Load every member's outputs

In [ ]:
manifest = pd.read_csv(PATHS.processed_dir / "invoice_manifest.csv") if (PATHS.processed_dir / "invoice_manifest.csv").exists() else pd.DataFrame()

stamp_sig_predictions = pd.read_csv(PATHS.predictions_dir / "stamp_signature_predictions.csv") if (PATHS.predictions_dir / "stamp_signature_predictions.csv").exists() else pd.DataFrame()
region_predictions = pd.read_csv(PATHS.predictions_dir / "region_predictions.csv") if (PATHS.predictions_dir / "region_predictions.csv").exists() else pd.DataFrame()
region_iou_metrics = json.loads((PATHS.metrics_dir / "region_iou_metrics.json").read_text()) if (PATHS.metrics_dir / "region_iou_metrics.json").exists() else {}
stamp_signature_metrics = json.loads((PATHS.metrics_dir / "stamp_signature_metrics.json").read_text()) if (PATHS.metrics_dir / "stamp_signature_metrics.json").exists() else {}

ocr_outputs = pd.read_csv(PATHS.predictions_dir / "ocr_outputs.csv") if (PATHS.predictions_dir / "ocr_outputs.csv").exists() else pd.DataFrame()
parameter_presence_results = pd.read_csv(PATHS.predictions_dir / "parameter_presence_results.csv") if (PATHS.predictions_dir / "parameter_presence_results.csv").exists() else pd.DataFrame()
terms_extraction_results = pd.read_csv(PATHS.predictions_dir / "terms_extraction_results.csv") if (PATHS.predictions_dir / "terms_extraction_results.csv").exists() else pd.DataFrame()

print("Loaded:", {
    "manifest_rows": len(manifest), "stamp_sig_rows": len(stamp_sig_predictions),
    "region_rows": len(region_predictions), "ocr_rows": len(ocr_outputs),
    "parameter_rows": len(parameter_presence_results), "terms_rows": len(terms_extraction_results),
})


## 3. Build final JSON per invoice

In [ ]:
from src.final_json_builder import build_final_json

document_ids = sorted(set(manifest.get("document_id", pd.Series(dtype=str))))
PATHS.final_json_dir.mkdir(parents=True, exist_ok=True)

final_records = {}
for doc_id in document_ids:
    stamp_sig_rows = stamp_sig_predictions[stamp_sig_predictions.get("document_id", pd.Series(dtype=str)) == doc_id].to_dict("records") if not stamp_sig_predictions.empty else []
    region_rows = region_predictions[region_predictions.get("document_id", pd.Series(dtype=str)) == doc_id].to_dict("records") if not region_predictions.empty else []
    parameter_rows = parameter_presence_results[parameter_presence_results.get("document_id", pd.Series(dtype=str)) == doc_id].to_dict("records") if not parameter_presence_results.empty else []

    terms_row = terms_extraction_results[terms_extraction_results.get("document_id", pd.Series(dtype=str)) == doc_id]
    if not terms_row.empty:
        r = terms_row.iloc[0]
        payment_context = {"invoice_date": r.get("invoice_date"), "due_date": r.get("due_date"), "payment_terms": r.get("payment_terms"), "billing_due_days": r.get("billing_due_days")}
        terms_and_conditions = {
            "region_detected": bool(r.get("extracted_text")), "late_payment_clause_detected": bool(r.get("late_payment_flag")),
            "dispute_clause_detected": bool(r.get("dispute_flag")), "penalty_clause_detected": bool(r.get("penalty_flag")),
            "extracted_text": r.get("extracted_text", ""), "summary": r.get("summary", ""),
        }
    else:
        payment_context = {"invoice_date": None, "due_date": None, "payment_terms": None, "billing_due_days": None}
        terms_and_conditions = {"region_detected": False, "late_payment_clause_detected": False, "dispute_clause_detected": False, "penalty_clause_detected": False, "extracted_text": "", "summary": ""}

    model_metrics = {
        "region_mean_iou": region_iou_metrics.get("overall_mean_iou"),
        "stamp_iou": stamp_signature_metrics.get("stamp", {}).get("mean_iou"),
        "signature_iou": stamp_signature_metrics.get("signature", {}).get("mean_iou"),
    }

    record = build_final_json(
        document_id=doc_id, source_image=doc_id,
        stamp_sig_rows=stamp_sig_rows, region_rows=region_rows, parameter_rows=parameter_rows,
        payment_context=payment_context, terms_and_conditions=terms_and_conditions, model_metrics=model_metrics,
    )
    final_records[doc_id] = record
    (PATHS.final_json_dir / f"{doc_id}.json").write_text(json.dumps(record, indent=2, default=str), encoding="utf-8")

print(f"Wrote {len(final_records)} final JSON records to {PATHS.final_json_dir}")


## 4. Final pipeline report

In [ ]:
ready_count = sum(1 for r in final_records.values() if r["pistacio_readiness"]["can_create_digital_obligation_record"])
report_lines = [
    "# Final Pipeline Report",
    "",
    f"- Invoices processed: {len(final_records)}",
    f"- Pistac.io-ready: {ready_count} / {len(final_records)}",
    f"- Region mean IoU: {region_iou_metrics.get('overall_mean_iou')}",
    f"- Stamp IoU: {stamp_signature_metrics.get('stamp', {}).get('mean_iou')}",
    f"- Signature IoU: {stamp_signature_metrics.get('signature', {}).get('mean_iou')}",
]
final_report_text = "\n".join(report_lines)
print(final_report_text)


## 5. Streamlit app + presentation check

In [ ]:
app_path = PATHS.repo_root / "app" / "streamlit_app.py"
demo_script_path = PATHS.repo_root / "presentation" / "demo_script.md"
print("streamlit_app.py exists:", app_path.exists())
print("demo_script.md exists:", demo_script_path.exists())
print("\nTo launch the demo: streamlit run app/streamlit_app.py")


In [ ]:
# --- Final export cell ---
# Save every output required by model_interface_contract.md

PATHS.reports_dir.mkdir(parents=True, exist_ok=True)
(PATHS.reports_dir / "final_pipeline_report.md").write_text(final_report_text, encoding="utf-8")

member_out = PATHS.member_outputs_dir("hessam_pm_integration")
member_out.mkdir(parents=True, exist_ok=True)
(member_out / "final_pipeline_report.md").write_text(final_report_text, encoding="utf-8")

print('Export complete.')
